## Gradiente de SINGX

Para cada bloque $j=1,\dots,m$, con $y^{(j)}=(x_{4j-3},x_{4j-2},x_{4j-1},x_{4j})^t$, definimos

$$u_1^{(j)}=x_{4j-3}+10x_{4j-2},\quad u_2^{(j)}=x_{4j-1}-x_{4j},\quad u_3^{(j)}=x_{4j-2}-2x_{4j-1},\quad u_4^{(j)}=x_{4j-3}-x_{4j}.$$

Entonces

$$f(x)=\sum_{j=1}^{m}\Big[(u_1^{(j)})^2+5(u_2^{(j)})^2+(u_3^{(j)})^4+10(u_4^{(j)})^4\Big],$$

y aplicando la regla de la cadena se obtiene, para cada bloque,

$$\frac{\partial f}{\partial x_{4j-3}}=2u_1^{(j)}+40(u_4^{(j)})^3,\qquad \frac{\partial f}{\partial x_{4j-2}}=20u_1^{(j)}+4(u_3^{(j)})^3,$$

$$\frac{\partial f}{\partial x_{4j-1}}=10u_2^{(j)}-8(u_3^{(j)})^3,\qquad \frac{\partial f}{\partial x_{4j}}=-10u_2^{(j)}-40(u_4^{(j)})^3.$$

En forma matricial, $\nabla_{y^{(j)}}f = A^t\nabla_u\big((u_1)^2+5(u_2)^2+(u_3)^4+10(u_4)^4\big)$, con $A$ la matriz invertible del análisis de existencia.

In [14]:
import numpy as np

def singx_f(x):
    """
    Evalúa la función SINGX (Powell singular extendida).
    x: array de longitud n (múltiplo de 4).
    """
    n = len(x)
    m = n // 4
    f = 0.0
    for j in range(m):
        u1 = x[4*j]     + 10.0*x[4*j+1]
        u2 = x[4*j+2]   - x[4*j+3]
        u3 = x[4*j+1]   - 2.0*x[4*j+2]
        u4 = x[4*j]     - x[4*j+3]
        f += u1**2 + 5.0*u2**2 + u3**4 + 10.0*u4**4
    return f


def singx_grad(x):
    """
    Calcula el gradiente analítico de la función SINGX.
    Devuelve un array de la misma dimensión que x.
    """
    n = len(x)
    m = n // 4
    g = np.zeros(n)
    for j in range(m):
        u1 = x[4*j]     + 10.0*x[4*j+1]
        u2 = x[4*j+2]   - x[4*j+3]
        u3 = x[4*j+1]   - 2.0*x[4*j+2]
        u4 = x[4*j]     - x[4*j+3]
        g[4*j]     = 2.0*u1 + 40.0*u4**3
        g[4*j+1]   = 20.0*u1 + 4.0*u3**3
        g[4*j+2]   = 10.0*u2 - 8.0*u3**3
        g[4*j+3]   = -10.0*u2 - 40.0*u4**3
    return g

# Verificación rápida con n=4, x0=(1,1,1,1)
x0 = np.array([1.0, 1.0, 1.0, 1.0])
print("f(x0) =", singx_f(x0))          # debe dar 122.0
print("grad(x0) =", singx_grad(x0))    # debe dar [22, 216, 8, 0]
print("||grad(x0)|| =", np.linalg.norm(singx_grad(x0)))  # sqrt(47204)

f(x0) = 122.0
grad(x0) = [ 22. 216.   8.  -0.]
||grad(x0)|| = 217.26481537515457


## Solución práctica: Método de Máximo Descenso con búsqueda lineal de Armijo

El algoritmo se implementa numéricamente con una búsqueda lineal de Armijo (o Wolfe). Dado $x_0\in\mathbb{R}^n$ y tolerancia $\varepsilon>0$:

1. Calcular $g_k=\nabla f(x_k)$. Si $\|g_k\|<\varepsilon$, parar.
2. Dirección: $d_k=-g_k/\|g_k\|$.
3. Búsqueda lineal: encontrar $\alpha_k$ que satisfaga Armijo
   $f(x_k+\alpha_k d_k)\le f(x_k)+c_1\alpha_k g_k^t d_k$.
4. Actualizar $x_{k+1}=x_k+\alpha_k d_k$, $k\leftarrow k+1$, volver al paso 1.

**Comentario sobre SINGX.** La matriz Hessiana de SINGX es singular en el óptimo $x^*=0$ (rango $2m<n$), por lo que el Teorema 9.2.2 no garantiza una razón de convergencia lineal acotada por $(\lambda-1)^2/(\lambda+1)^2$. En la práctica, el método requiere del orden de $10^3$–$10^4$ iteraciones para reducir el gradiente a $10^{-6}$, y la sucesión exhibe el patrón de zigzag ($d_k^t d_{k+1}\approx 0$) descrito en el Capítulo 9.

In [15]:
def max_descenso(f, grad, x0, tol=1e-6, max_iter=10000,
                 c1=1e-4, rho=0.5, verbose=True):
    """
    Método de Máximo Descenso con búsqueda lineal de Armijo (backtracking).

    Parámetros
    ----------
    f    : función objetivo
    grad : gradiente analítico
    x0   : punto inicial
    tol  : tolerancia sobre ||grad||
    c1   : constante de Armijo (típicamente 1e-4)
    rho  : factor de reducción del paso (típicamente 0.5)

    Devuelve
    --------
    x_opt   : aproximación al mínimo
    history : diccionario con listas de iterados, valores de f y ||grad||
    """
    x = x0.copy().astype(float)
    history = {'x': [x.copy()],
               'f': [f(x)],
               'gnorm': [np.linalg.norm(grad(x))]}

    for k in range(max_iter):
        g = grad(x)
        gnorm = np.linalg.norm(g)

        if gnorm < tol:
            if verbose:
                print(f"Convergencia en {k} iteraciones, ||g|| = {gnorm:.2e}")
            break

        d = -g / gnorm                       # dirección de máximo descenso
        fk = f(x)
        gd = g @ d                           # = -||g|| < 0
        alpha = 1.0                          # paso inicial

        # Backtracking de Armijo
        while f(x + alpha*d) > fk + c1*alpha*gd:
            alpha *= rho

        x = x + alpha*d
        history['x'].append(x.copy())
        history['f'].append(f(x))
        history['gnorm'].append(np.linalg.norm(grad(x)))

        if verbose and (k % 100 == 0 or k < 5):
            print(f"k={k:4d}  f={history['f'][-1]:.6e}  "
                  f"||g||={history['gnorm'][-1]:.3e}  alpha={alpha:.3e}")

    return x, history

In [16]:
# Ejemplo con n=4, x0=(1,1,1,1): primera iteración 122 -> ~14.43
x0 = np.array([1.0, 1.0, 1.0, 1.0])
print(f"f(x0) = {singx_f(x0):.4f}")

x_opt, hist = max_descenso(singx_f, singx_grad, x0, tol=1e-6, max_iter=5000)

print(f"\nf(x_opt)  = {singx_f(x_opt):.6e}")
print(f"x_opt     = {x_opt}")
print(f"Iteraciones ejecutadas: {len(hist['f'])-1}")
print(f"f tras 1ª iteración: {hist['f'][1]:.4f}")

f(x0) = 122.0000
k=   0  f=1.452831e+01  ||g||=5.708e+01  alpha=1.000e+00
k=   1  f=1.159343e+01  ||g||=5.300e+01  alpha=1.000e+00
k=   2  f=8.712969e+00  ||g||=4.789e+01  alpha=5.000e-01
k=   3  f=2.772921e+00  ||g||=1.071e+01  alpha=2.500e-01
k=   4  f=1.306090e+00  ||g||=1.239e+01  alpha=2.500e-01
k= 100  f=6.788734e-03  ||g||=7.933e-01  alpha=7.812e-03
k= 200  f=3.865595e-03  ||g||=7.915e-01  alpha=7.812e-03
k= 300  f=2.830556e-03  ||g||=7.909e-01  alpha=7.812e-03
k= 400  f=2.355088e-03  ||g||=7.905e-01  alpha=7.812e-03
k= 500  f=4.128494e-06  ||g||=7.706e-04  alpha=7.629e-06
k= 600  f=4.004475e-06  ||g||=7.706e-04  alpha=7.629e-06
k= 700  f=3.885977e-06  ||g||=7.706e-04  alpha=7.629e-06
k= 800  f=3.772672e-06  ||g||=7.706e-04  alpha=7.629e-06
k= 900  f=3.664261e-06  ||g||=7.706e-04  alpha=7.629e-06
k=1000  f=3.560464e-06  ||g||=7.706e-04  alpha=7.629e-06
k=1100  f=3.461022e-06  ||g||=7.706e-04  alpha=7.629e-06
k=1200  f=3.365693e-06  ||g||=7.706e-04  alpha=7.629e-06
k=1300  f=3.27

# Método de Newton aplicado a SINGX

## Gradiente y Hessiana de SINGX

Para cada bloque $j=1,\dots,m$, con $y^{(j)}=(x_{4j-3},x_{4j-2},x_{4j-1},x_{4j})^t$, definimos
$$u^{(j)}=A\,y^{(j)},\qquad A=\begin{pmatrix}1&10&0&0\\0&0&1&-1\\0&1&-2&0\\1&0&0&-1\end{pmatrix},$$
es decir, $u_1=x_{4j-3}+10x_{4j-2}$, $u_2=x_{4j-1}-x_{4j}$, $u_3=x_{4j-2}-2x_{4j-1}$, $u_4=x_{4j-3}-x_{4j}$.

Entonces
$$f(x)=\sum_{j=1}^{m}\big[u_1^2+5u_2^2+u_3^4+10u_4^4\big].$$

**Gradiente.** Aplicando la regla de la cadena:
$$\nabla_{y^{(j)}}f=A^t\nabla_u\phi(u^{(j)}),\qquad \phi(u)=u_1^2+5u_2^2+u_3^4+10u_4^4,$$
con $\nabla_u\phi=(2u_1,\,10u_2,\,4u_3^3,\,40u_4^3)^t$. Componente a componente:
$$\frac{\partial f}{\partial x_{4j-3}}=2u_1+40u_4^3,\quad \frac{\partial f}{\partial x_{4j-2}}=20u_1+4u_3^3,\quad \frac{\partial f}{\partial x_{4j-1}}=10u_2-8u_3^3,\quad \frac{\partial f}{\partial x_{4j}}=-10u_2-40u_4^3.$$

**Hessiana.** Como $\nabla_u^2\phi=\mathrm{diag}(2,\,10,\,12u_3^2,\,120u_4^2)$, se tiene
$$\nabla_{y^{(j)}}^2 f=A^t\,\mathrm{diag}(2,\,10,\,12u_3^2,\,120u_4^2)\,A.$$
La Hessiana global es bloque diagonal, con un bloque $4\times 4$ por cada $j$.

## Caso 1: Hessiana singular en $x_0=(1,1,1,1)^t$

En $x_0=(1,1,1,1)^t$ se tiene $u=(11,0,-1,0)$ y por tanto $u_4=0$, lo que anula la cuarta entrada de la diagonal $\mathrm{diag}(2,10,12,0)$ y hace que $\nabla^2 f(x_0)$ tenga rango $3<4$. **El método de Newton puro no está bien definido en este punto** (el sistema $\nabla^2 f(x_0)\,d_0=-\nabla f(x_0)$ no tiene solución única). Esto motiva el uso de un punto inicial alternativo o de regularización.

## Caso 2: Hessiana no singular en $x_0=(1,1,1,0)^t$

En $x_0=(1,1,1,0)^t$: $u=(11,1,-1,1)$, $u_4=1\neq 0$, luego $\nabla^2 f(x_0)$ es definida positiva (rango completo). Se tiene
$$f(x_0)=11^2+5\cdot 1^2+(-1)^4+10\cdot 1^4=137,$$
$$\nabla f(x_0)=(62,\,216,\,18,\,-50)^t,\qquad \|\nabla f(x_0)\|\approx 230.92,$$
$$\nabla^2 f(x_0)=\begin{pmatrix}122&20&0&-120\\20&212&-24&0\\0&-24&58&-10\\-120&0&-10&130\end{pmatrix}.$$
Se resuelve $\nabla^2 f(x_0)\,d_0=-\nabla f(x_0)$ y se actualiza $x_1=x_0+d_0$. La convergencia es cuadrática localmente (Teorema 9.3.1). En las proximidades de $x^*=0$, la Hessiana vuelve a ser singular (rango $2m$), por lo que el método puro pierde su garantía teórica y se requiere regularización $[\nabla^2 f(x_k)+\varepsilon_k I]$.

In [17]:
import numpy as np

A = np.array([[1., 10., 0., 0.],
              [0., 0., 1., -1.],
              [0., 1., -2., 0.],
              [1., 0., 0., -1.]])

def singx_f(x):
    """Función SINGX (Powell singular extendida)."""
    n = len(x); m = n // 4
    f = 0.0
    for j in range(m):
        u = A @ x[4*j:4*j+4]
        f += u[0]**2 + 5*u[1]**2 + u[2]**4 + 10*u[3]**4
    return f

def singx_grad(x):
    """Gradiente analítico de SINGX."""
    n = len(x); m = n // 4
    g = np.zeros(n)
    for j in range(m):
        y = x[4*j:4*j+4]
        u = A @ y
        gu = np.array([2*u[0], 10*u[1], 4*u[2]**3, 40*u[3]**3])
        g[4*j:4*j+4] = A.T @ gu
    return g

def singx_hess(x):
    """Hessiana analítica de SINGX (bloque diagonal)."""
    n = len(x); m = n // 4
    H = np.zeros((n, n))
    for j in range(m):
        u = A @ x[4*j:4*j+4]
        D = np.diag([2., 10., 12.*u[2]**2, 120.*u[3]**2])
        H[4*j:4*j+4, 4*j:4*j+4] = A.T @ D @ A
    return H

def newton_method(f, grad, hess, x0, tol=1e-10, max_iter=100, verbose=True):
    """Método de Newton puro para minimización irrestricta."""
    x = x0.copy().astype(float)
    hist = {'x': [x.copy()], 'f': [f(x)], 'gnorm': [np.linalg.norm(grad(x))]}
    for k in range(max_iter):
        g = grad(x); gn = np.linalg.norm(g)
        if gn < tol:
            if verbose: print(f"Convergencia en {k} iteraciones, ||g||={gn:.2e}")
            break
        H = hess(x)
        try:
            d = np.linalg.solve(H, -g)
        except np.linalg.LinAlgError:
            if verbose: print(f"Hessiana singular en k={k}, se detiene el método")
            break
        x = x + d
        hist['x'].append(x.copy()); hist['f'].append(f(x)); hist['gnorm'].append(np.linalg.norm(grad(x)))
        if verbose:
            print(f"k={k}: f={hist['f'][-1]:.6e}, ||g||={hist['gnorm'][-1]:.3e}, ||d||={np.linalg.norm(d):.3e}")
    return x, hist

In [18]:
# ----- Caso 1: Hessiana singular en x0 = (1,1,1,1) -----
x0 = np.array([1., 1., 1., 1.])
print(f"Caso 1: x0 = {x0}")
print(f"f(x0)      = {singx_f(x0):.4f}")          # esperado: 122.0
print(f"||grad||   = {np.linalg.norm(singx_grad(x0)):.4f}")  # esperado: ≈ 217.26
print(f"rang(H)    = {np.linalg.matrix_rank(singx_hess(x0))}")  # esperado: 3 (singular)
print(f"det(H)     = {np.linalg.det(singx_hess(x0)):.3e}")  # esperado: ≈ 0
# Newton fallará aquí por Hessiana singular
x_sing, _ = newton_method(singx_f, singx_grad, singx_hess, x0, verbose=True)

Caso 1: x0 = [1. 1. 1. 1.]
f(x0)      = 122.0000
||grad||   = 217.2648
rang(H)    = 3
det(H)     = 2.478e-11
k=0: f=1.095194e+04, ||g||=1.077e+04, ||d||=6.207e+00
k=1: f=2.163347e+03, ||g||=3.191e+03, ||d||=1.751e+00
k=2: f=4.273277e+02, ||g||=9.455e+02, ||d||=1.168e+00
k=3: f=8.441042e+01, ||g||=2.801e+02, ||d||=7.784e-01
k=4: f=1.667366e+01, ||g||=8.300e+01, ||d||=5.189e-01
k=5: f=3.293563e+00, ||g||=2.459e+01, ||d||=3.459e-01
k=6: f=6.505804e-01, ||g||=7.287e+00, ||d||=2.306e-01
k=7: f=1.285097e-01, ||g||=2.159e+00, ||d||=1.538e-01
k=8: f=2.538463e-02, ||g||=6.397e-01, ||d||=1.025e-01
k=9: f=5.014248e-03, ||g||=1.895e-01, ||d||=6.833e-02
k=10: f=9.904688e-04, ||g||=5.616e-02, ||d||=4.556e-02
k=11: f=1.956482e-04, ||g||=1.664e-02, ||d||=3.037e-02
k=12: f=3.864655e-05, ||g||=4.931e-03, ||d||=2.025e-02
k=13: f=7.633886e-06, ||g||=1.461e-03, ||d||=1.350e-02
k=14: f=1.507928e-06, ||g||=4.329e-04, ||d||=8.999e-03
k=15: f=2.978624e-07, ||g||=1.283e-04, ||d||=5.999e-03
k=16: f=5.883701e-08,

In [19]:
# ----- Caso 2: Hessiana NO singular en x0 = (1,1,1,0) -----
x0 = np.array([1., 1., 1., 0.])
print(f"\nCaso 2: x0 = {x0}")
print(f"f(x0)      = {singx_f(x0):.4f}")          # esperado: 137.0
print(f"||grad||   = {np.linalg.norm(singx_grad(x0)):.4f}")  # esperado: ≈ 230.92
print(f"rang(H)    = {np.linalg.matrix_rank(singx_hess(x0))}")  # esperado: 4 (no singular)
print(f"autovalores H = {np.linalg.eigvalsh(singx_hess(x0))}")  # todos > 0
print()
x_opt, hist = newton_method(singx_f, singx_grad, singx_hess, x0, tol=1e-10, max_iter=50)
print(f"\nf(x_opt)   = {singx_f(x_opt):.2e}")   # esperado: → 0
print(f"x_opt      = {x_opt}")                    # esperado: → (0,0,0,0)
print(f"Iteraciones: {len(hist['f'])-1}")
# Observación: la primera iteración reduce f de 137 a un valor pequeño;
# a partir de ahí la convergencia es cuadrática hasta llegar cerca de 0,
# donde la Hessiana vuelve a ser singular.


Caso 2: x0 = [1. 1. 1. 0.]
f(x0)      = 137.0000
||grad||   = 230.9199
rang(H)    = 4
autovalores H = [  4.38736996  54.33684452 212.14389897 251.13188655]

k=0: f=2.172840e+00, ||g||=1.697e+01, ||d||=1.339e+00
k=1: f=4.292029e-01, ||g||=5.028e+00, ||d||=3.463e-01
k=2: f=8.478081e-02, ||g||=1.490e+00, ||d||=2.309e-01
k=3: f=1.674683e-02, ||g||=4.414e-01, ||d||=1.539e-01
k=4: f=3.308015e-03, ||g||=1.308e-01, ||d||=1.026e-01
k=5: f=6.534351e-04, ||g||=3.875e-02, ||d||=6.841e-02
k=6: f=1.290736e-04, ||g||=1.148e-02, ||d||=4.560e-02
k=7: f=2.549602e-05, ||g||=3.402e-03, ||d||=3.040e-02
k=8: f=5.036251e-06, ||g||=1.008e-03, ||d||=2.027e-02
k=9: f=9.948150e-07, ||g||=2.987e-04, ||d||=1.351e-02
k=10: f=1.965067e-07, ||g||=8.850e-05, ||d||=9.008e-03
k=11: f=3.881613e-08, ||g||=2.622e-05, ||d||=6.006e-03
k=12: f=7.667384e-09, ||g||=7.769e-06, ||d||=4.004e-03
k=13: f=1.514545e-09, ||g||=2.302e-06, ||d||=2.669e-03
k=14: f=2.991694e-10, ||g||=6.821e-07, ||d||=1.779e-03
k=15: f=5.909519e-11, ||g||

## Gradiente de Brown Almost Linear (ALMOST)

La función **ALMOST** (Brown Almost Linear) se define como

$$f(x) = \sum_{i=1}^{n-1}\Big(x_i + S - (n+1)\Big)^2 + \Big(P - 1\Big)^2,$$

donde
$$S = \sum_{j=1}^{n} x_j, \qquad P = \prod_{j=1}^{n} x_j.$$

Definimos los residuos
$$r_i = x_i + S - (n+1)\quad (i=1,\dots,n-1), \qquad r_n = P - 1,$$
y denotamos $R = \sum_{i=1}^{n-1} r_i$.

Aplicando la regla de la cadena, para $k=1,\dots,n-1$ se cumple
$$\frac{\partial r_i}{\partial x_k} = \delta_{ik} + 1 \quad (i=1,\dots,n-1), \qquad \frac{\partial r_n}{\partial x_k} = P_k := \prod_{j\neq k} x_j,$$
y por tanto
$$\frac{\partial f}{\partial x_k} = 2\big(R + r_k\big) + 2\,r_n\,P_k \quad (k=1,\dots,n-1),$$
$$\frac{\partial f}{\partial x_n} = 2R + 2\,r_n\,P_n.$$

**Observación.** El término $P_k = P/x_k$ (producto de todas las variables excepto $x_k$) introduce un acoplamiento global entre todas las coordenadas: un cambio en una variable afecta a los $n-1$ residuos lineales y al término del producto. Esto genera valles curvos y estrechos donde el antigradiente no apunta directamente al mínimo.

In [20]:
import numpy as np

def almost_f(x):
    """
    Evalúa la función Brown Almost Linear (ALMOST).
    x: array de longitud n ≥ 1.
    """
    n = len(x)
    S = np.sum(x)
    r = x[:n-1] + S - (n+1)        # primeros n-1 residuos
    prod = np.prod(x)
    return np.sum(r**2) + (prod - 1.0)**2


def almost_grad(x):
    """
    Gradiente analítico de ALMOST.
    Devuelve un array de la misma dimensión que x.
    """
    n = len(x)
    S = np.sum(x)
    r = np.zeros(n)
    r[:n-1] = x[:n-1] + S - (n+1)   # residuos lineales
    prod = np.prod(x)
    r[n-1] = prod - 1.0              # residuo del producto
    R = np.sum(r[:n-1])

    g = np.zeros(n)
    for k in range(n):
        # P_k = producto de todos los x_j con j != k (sin dividir por cero)
        if x[k] != 0:
            Pk = prod / x[k]
        else:
            Pk = np.prod(np.delete(x, k))

        if k < n-1:
            g[k] = 2.0 * (R + r[k]) + 2.0 * r[n-1] * Pk
        else:
            g[k] = 2.0 * R + 2.0 * r[n-1] * Pk
    return g

# ----- Verificación con n=4, x0=(0.5,0.5,0.5,0.5) -----
x0 = np.array([0.5, 0.5, 0.5, 0.5])
print(f"f(x0)       = {almost_f(x0):.8f}")            # esperado: 19.62890625
print(f"grad(x0)    = {almost_grad(x0)}")             # esperado: (-20.234375, -20.234375, -20.234375, -15.234375)
print(f"||grad(x0)||= {np.linalg.norm(almost_grad(x0)):.6f}")  # esperado: ≈ 38.2153

f(x0)       = 19.62890625
grad(x0)    = [-20.234375 -20.234375 -20.234375 -15.234375]
||grad(x0)||= 38.214866


## Solución práctica: Máximo Descenso con búsqueda lineal de Armijo

El algoritmo de máximo descenso con retroceso de Armijo se implementa exactamente igual que para SINGX (ver notebook de SINGX), ya que es genérico para cualquier par $(f,\nabla f)$.

**Resultados esperados para $n=4$, $x_0=(0.5,0.5,0.5,0.5)$:**
- $f(x_0) = 19.62890625$, $\|\nabla f(x_0)\| \approx 38.2153$.
- $d_0 \approx (0.5295,\,0.5295,\,0.5295,\,0.3987)$.
- La primera iteración **acepta $\alpha_0=1$** y produce $x_1\approx(1.0295,1.0295,1.0295,0.8986)$ con $f(x_1)\approx 1.2\times 10^{-3}$.
- A partir de la segunda iteración, $\alpha_k$ cae drásticamente (p. ej.\ $\alpha_1\approx 3.9\times 10^{-3}$) porque $d_1$ es casi ortogonal a la dirección que lleva al mínimo: **patrón de zigzag**.
- La sucesión converge a $x^*=(1,1,1,1)$ con $f(x^*)=0$, pero de forma lenta por el acoplamiento global de la función.

In [21]:
def max_descenso(f, grad, x0, tol=1e-6, max_iter=20000,
                 c1=1e-4, rho=0.5, verbose=True, print_every=10):
    """
    Método de Máximo Descenso con búsqueda lineal de Armijo (backtracking).

    Devuelve
    --------
    x_opt   : aproximación al mínimo
    history : diccionario con listas de iterados, valores de f, ||grad||, alpha
    """
    x = x0.copy().astype(float)
    history = {'x': [x.copy()],
               'f': [f(x)],
               'gnorm': [np.linalg.norm(grad(x))],
               'alpha': []}

    for k in range(max_iter):
        g = grad(x)
        gnorm = np.linalg.norm(g)

        if gnorm < tol:
            if verbose:
                print(f"Convergencia en {k} iteraciones, ||g|| = {gnorm:.2e}")
            break

        d = -g / gnorm                 # dirección de máximo descenso
        fk = f(x)
        gd = g @ d                     # = -||g|| < 0
        alpha = 1.0

        # Backtracking de Armijo
        while f(x + alpha*d) > fk + c1*alpha*gd:
            alpha *= rho

        x = x + alpha*d
        history['x'].append(x.copy())
        history['f'].append(f(x))
        history['gnorm'].append(np.linalg.norm(grad(x)))
        history['alpha'].append(alpha)

        if verbose and (k < 5 or k % print_every == 0):
            print(f"k={k:4d}  f={history['f'][-1]:.6e}  "
                  f"||g||={history['gnorm'][-1]:.3e}  alpha={alpha:.3e}")

    return x, history

In [22]:
# --------- Ejecución del Método de Máximo Descenso sobre ALMOST ---------
x0 = np.array([0.5, 0.5, 0.5, 0.5])

print(f"x0          = {x0}")
print(f"f(x0)       = {almost_f(x0):.8f}")   # 19.62890625
print(f"||grad(x0)||= {np.linalg.norm(almost_grad(x0)):.6f}")  # ≈ 38.2153
print()

x_opt, hist = max_descenso(almost_f, almost_grad, x0,
                           tol=1e-8, max_iter=20000,
                           c1=1e-4, rho=0.5,
                           verbose=True, print_every=10)

print(f"\n--- Resultado final ---")
print(f"x_opt     = {x_opt}")
print(f"f(x_opt)  = {almost_f(x_opt):.6e}")
print(f"||grad||  = {np.linalg.norm(almost_grad(x_opt)):.3e}")
print(f"Iteraciones totales: {len(hist['f'])-1}")
print(f"\nPrimeras iteraciones:")
for k in range(min(6, len(hist['f']))):
    print(f"  k={k:2d}  f={hist['f'][k]:.6e}  ||g||={hist['gnorm'][k]:.3e}")

x0          = [0.5 0.5 0.5 0.5]
f(x0)       = 19.62890625
||grad(x0)||= 38.214866

k=   0  f=1.207021e-03  ||g||=1.754e-01  alpha=1.000e+00
k=   1  f=8.647360e-04  ||g||=1.855e-02  alpha=3.906e-03
k=   2  f=6.681006e-04  ||g||=1.552e-01  alpha=3.125e-02
k=   3  f=4.078252e-04  ||g||=2.546e-02  alpha=3.906e-03
k=   4  f=4.003681e-04  ||g||=2.041e-02  alpha=9.766e-04
k=  10  f=3.690106e-04  ||g||=2.219e-02  alpha=9.766e-04
k=  20  f=3.227262e-04  ||g||=2.223e-02  alpha=9.766e-04
k=  30  f=2.834631e-04  ||g||=2.225e-02  alpha=9.766e-04
k=  40  f=2.499606e-04  ||g||=2.226e-02  alpha=9.766e-04
k=  50  f=2.212170e-04  ||g||=2.227e-02  alpha=9.766e-04
k=  60  f=1.964344e-04  ||g||=2.228e-02  alpha=9.766e-04
k=  70  f=1.749705e-04  ||g||=2.229e-02  alpha=9.766e-04
k=  80  f=1.563043e-04  ||g||=2.230e-02  alpha=9.766e-04
k=  90  f=1.400100e-04  ||g||=2.231e-02  alpha=9.766e-04
k= 100  f=1.257366e-04  ||g||=2.231e-02  alpha=9.766e-04
k= 110  f=1.131936e-04  ||g||=2.232e-02  alpha=9.766e-04
k= 12

# Método de Newton aplicado a Brown Almost Linear (ALMOST)

## Gradiente de ALMOST

La función **ALMOST** es
$$f(x)=\sum_{i=1}^{n-1}\big(x_i+S-(n+1)\big)^2+(P-1)^2,\qquad S=\sum_{j=1}^n x_j,\quad P=\prod_{j=1}^n x_j.$$
Definimos los residuos $r_i=x_i+S-(n+1)$ para $i=1,\dots,n-1$, $r_n=P-1$, y denotamos $R=\sum_{i=1}^{n-1}r_i$. Entonces
$$\frac{\partial f}{\partial x_k}=2(R+r_k)+2r_n P_k\quad(k=1,\dots,n-1),\qquad\frac{\partial f}{\partial x_n}=2R+2r_n P_n,$$
donde $P_k=\prod_{j\neq k}x_j$.

## Hessiana de ALMOST

Aplicando la regla de la cadena y teniendo en cuenta que $\partial r_i/\partial x_k=\delta_{ik}+1$ y $\partial^2 r_n/\partial x_k\partial x_l=0$ si $k=l$, $=P_{kl}:=\prod_{j\notin\{k,l\}}x_j$ si $k\neq l$, se obtiene:
$$H_{kl}=2\sum_{i=1}^{n-1}(\delta_{ik}+1)(\delta_{il}+1)+2P_kP_l+2r_n\cdot\begin{cases}P_{kl}, & k\neq l\\ 0, & k=l\end{cases}$$
Evaluando la suma:
- Si $k,l\le n-1$: $H_{kl}=2(n+1+\delta_{kl})+2P_kP_l+2r_nP_{kl}\cdot\mathbb{1}_{k\neq l}$.
- Si $k\le n-1$, $l=n$: $H_{kl}=2n+2P_kP_n+2r_nP_{kl}$.
- $H_{nn}=2(n-1)+2P_n^2$.

## Ejemplo de una iteración ($n=4$, $x_0=(0.5,0.5,0.5,0.5)^t$)

$S=2$, $r_i=-2.5$ $(i=1,2,3)$, $R=-7.5$, $P=0.0625$, $r_4=-0.9375$, $P_k=0.125$ para todo $k$.
$$f(x_0)=3\cdot(-2.5)^2+(-0.9375)^2\approx 19.6289,$$
$$\nabla f(x_0)=(-20.2344,\,-20.2344,\,-20.2344,\,-15.2344)^t,\quad\|\nabla f(x_0)\|\approx 38.2153,$$
$$\nabla^2 f(x_0)=\begin{pmatrix}12.031&9.5625&9.5625&7.5625\\9.5625&12.031&9.5625&7.5625\\9.5625&9.5625&12.031&7.5625\\7.5625&7.5625&7.5625&6.031\end{pmatrix},$$
que es definida positiva. Se resuelve $\nabla^2 f(x_0)d_0=-\nabla f(x_0)$, se actualiza $x_1=x_0+d_0$ y se observa una reducción drástica de \(f\) (de \(19.63\) a valores del orden de \(10^{-2}\)). La convergencia es cuadrática en una vecindad de \(x^*=(1,1,1,1)\), donde \(\nabla^2 f(x^*)\) es definida positiva (autovalores \(2,\,2,\,0.0871,\,45.913\) para \(n=4\)).

In [23]:
def almost_f(x):
    """Función ALMOST."""
    n = len(x); S = np.sum(x)
    r = x[:n-1] + S - (n+1)
    return np.sum(r**2) + (np.prod(x) - 1.0)**2

def almost_grad(x):
    """Gradiente analítico de ALMOST."""
    n = len(x); S = np.sum(x); P = np.prod(x)
    r = np.zeros(n)
    r[:n-1] = x[:n-1] + S - (n+1)
    r[n-1] = P - 1.0
    R = np.sum(r[:n-1])
    Pk = np.array([P/x[k] if x[k] != 0 else np.prod(np.delete(x, k)) for k in range(n)])
    g = np.zeros(n)
    for k in range(n-1):
        g[k] = 2*(R + r[k]) + 2*r[n-1]*Pk[k]
    g[n-1] = 2*R + 2*r[n-1]*Pk[n-1]
    return g

def almost_hess(x):
    """Hessiana analítica de ALMOST."""
    n = len(x); S = np.sum(x); P = np.prod(x)
    r = np.zeros(n)
    r[:n-1] = x[:n-1] + S - (n+1)
    r[n-1] = P - 1.0
    rn = r[n-1]
    Pk = np.array([P/x[k] if x[k] != 0 else np.prod(np.delete(x, k)) for k in range(n)])
    H = np.zeros((n, n))
    for k in range(n-1):
        for l in range(n-1):
            H[k, l] = 2*(n + 1 + (1 if k == l else 0)) + 2*Pk[k]*Pk[l]
            if k != l:
                H[k, l] += 2*rn*np.prod(np.delete(x, [k, l]))
        H[k, n-1] = 2*n + 2*Pk[k]*Pk[n-1] + 2*rn*np.prod(np.delete(x, [k, n-1]))
        H[n-1, k] = H[k, n-1]
    H[n-1, n-1] = 2*(n-1) + 2*Pk[n-1]**2
    return H

In [24]:
# ----- Verificación con n=4, x0=(0.5,0.5,0.5,0.5) -----
x0 = np.array([0.5, 0.5, 0.5, 0.5])
print(f"f(x0)       = {almost_f(x0):.8f}")            # esperado: 19.62890625
print(f"grad(x0)    = {almost_grad(x0)}")             # esperado: (-20.234375,)*3 + (-15.234375,)
print(f"||grad(x0)||= {np.linalg.norm(almost_grad(x0)):.6f}")  # esperado: ≈ 38.2153
print(f"H(x0)       =\n{np.round(almost_hess(x0), 6)}")
print(f"autovalores = {np.linalg.eigvalsh(almost_hess(x0))}")  # todos > 0
print()

# Newton desde x0
x_opt, hist = newton_method(almost_f, almost_grad, almost_hess, x0,
                            tol=1e-12, max_iter=50, verbose=True)

print(f"\n--- Resultado final ---")
print(f"x_opt      = {x_opt}")                # esperado: → (1,1,1,1)
print(f"f(x_opt)   = {almost_f(x_opt):.2e}")  # esperado: → 0
print(f"||grad||   = {np.linalg.norm(almost_grad(x_opt)):.2e}")
print(f"Iteraciones: {len(hist['f'])-1}")

# Hessiana en el óptimo (debe ser definida positiva)
print(f"\nAutovalores H(1,1,1,1) = {np.linalg.eigvalsh(almost_hess(np.ones(4)))}")
# esperado: ≈ (0.0871, 2, 2, 45.913)  -> todos > 0

f(x0)       = 19.62890625
grad(x0)    = [-20.234375 -20.234375 -20.234375 -15.234375]
||grad(x0)||= 38.214866
H(x0)       =
[[12.03125  9.5625   9.5625   7.5625 ]
 [ 9.5625  12.03125  9.5625   7.5625 ]
 [ 9.5625   9.5625  12.03125  7.5625 ]
 [ 7.5625   7.5625   7.5625   6.03125]]
autovalores = [ 0.44462812  2.46875     2.46875    36.74287188]

k=0: f=6.276738e-02, ||g||=2.457e+00, ||d||=1.197e+00
k=1: f=3.337418e-04, ||g||=5.393e-02, ||d||=5.653e-02
k=2: f=8.077751e+00, ||g||=2.650e+01, ||d||=6.776e+00
k=3: f=3.273639e+00, ||g||=1.006e+01, ||d||=7.180e-01
k=4: f=1.692277e+00, ||g||=3.605e+00, ||d||=5.691e-01
k=5: f=1.174033e+00, ||g||=1.235e+00, ||d||=5.250e-01
k=6: f=1.032566e+00, ||g||=3.763e-01, ||d||=4.317e-01
k=7: f=1.004897e+00, ||g||=1.046e-01, ||d||=2.860e-01
k=8: f=1.000669e+00, ||g||=2.791e-02, ||d||=1.618e-01
k=9: f=1.000088e+00, ||g||=7.256e-03, ||d||=8.578e-02
k=10: f=1.000011e+00, ||g||=1.854e-03, ||d||=4.420e-02
k=11: f=1.000001e+00, ||g||=4.690e-04, ||d||=2.245e-02
k=12